# Data Acquisition & Methodology

All raw data for this project is sourced from two APIs. In this notebook only data from PokéApi is collected because automation was unnecessary for the other source.

## Data Sources
* **[PokéApi](https://pokeapi.co/):** The primary source for all Pokémon related data (stats, abilities, generations, names, moves, etc.)
* **[Smogon API](https://github.com/pkmn/smogon/blob/main/API.md):** Smogon is an unofficial Pokémon Community of competitive players. They have a battle simulator called Pokémon Showdown (https://pokemonshowdown.com/) The data available from this API are pulled from their servers, it contains competitive usage statistics alongside detailed information about Pokémons used in battles. More details will be given about this API and its contents in the Machine Learning Phase.

---

## Smogon API
**Target Endpoints:** `/stats` (`gen9ubers`, `gen9ubersuu`, `gen9ou`, `gen9uu`)

In this notebook, data from the Smogon API is **not** actively fetched via scripts. Since the scope was limited to four specific JSON files, automation was unnecessary for this source.

*Note: Since there were only 4 JSONs of interest they were manually downloaded and can be found at data/raw folder of the repository.*

---

## PokéApi
**Target Endpoints:** `move-damage-class`, `ability`, `type`, `pokemon`, `generation`, and `pokemon-species`.

Unlike the Smogon data, PokeApi collection is fully automated within this notebook.

### PokéApi Architecture
PokéApi utilizes a nested resource structure. A standard query to a base endpoint (e.g., `https://pokeapi.co/api/v2/{endpoint}/?limit={limit}`) returns a list of names and URLs, rather than the full data objects.

**The first resource of the `pokemon` endpoint as an example:**
```json
{
  "name": "bulbasaur",
  "url": "https://pokeapi.co/api/v2/pokemon/1/"
}
```

### Approach
To get information about bulbasaur the url needs to be followed to fetch the actual resource. Hence the fetch_and_structure_resource function was defined to handle this architecture. Luckily, for each endpoint this architecture is followed exactly thus a single function was enough to handle data collection from this API.

In [1]:
import requests
import json
import os
import time

# Setup save path
save_folder = '../data/raw'
if not os.path.exists(save_folder):
    os.makedirs(save_folder)

def fetch_and_structure_resource(endpoint, filename, limit=20):
    list_url = f"https://pokeapi.co/api/v2/{endpoint}/?limit={limit}"

    try:
        print(f"Fetching list for {endpoint}...")
        response = requests.get(list_url)
        response.raise_for_status()
        results = response.json().get('results', [])

        categorized_data = {}

        print(f"Downloading details...")
        for item in results:
            name = item['name']
            detail_url = item['url']

            detail_res = requests.get(detail_url)
            detail_res.raise_for_status()

            # Store the full JSON detail inside categorized data dictionary using the name as the key
            categorized_data[name] = detail_res.json()

            # to not get banned
            time.sleep(0.1)

        # Save to data/raw
        file_path = os.path.join(save_folder, filename)
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(categorized_data, f, indent=4)

        print(f"\nSuccess! Saved {len(categorized_data)} items to {file_path}")

    except Exception as e:
        print(f"Error: {e}")

In [2]:
# Run for Move Damage Classes
fetch_and_structure_resource('move-damage-class', 'move_damage_classes_raw.json')

Fetching list for move-damage-class...

Success! Saved 3 items to ../data/raw\move_damage_classes_raw.json


In [3]:
# Run for Abilities
fetch_and_structure_resource('ability', 'abilities_raw.json', 500)

Fetching list for ability...

Success! Saved 371 items to ../data/raw\abilities_raw.json


In [4]:
# Run for Types
fetch_and_structure_resource('type', 'types_raw.json', 100)

Fetching list for type...

Success! Saved 21 items to ../data/raw\types_raw.json


In [5]:
# Run for Pokemon
fetch_and_structure_resource('pokemon', 'pokemon_raw.json', 2000)

Fetching list for pokemon...

Success! Saved 1350 items to ../data/raw\pokemon_raw.json


In [6]:
# Run for Generations
fetch_and_structure_resource('generation', 'generations_raw.json')

Fetching list for generation...

Success! Saved 9 items to ../data/raw\generations_raw.json


In [7]:
# Run for Species
fetch_and_structure_resource('pokemon-species', 'species_raw.json', 2000)

Fetching list for pokemon-species...

Success! Saved 1025 items to ../data/raw\species_raw.json
